# TuRBO for any capstone function

Adapted from the course TuRBO tutorial (Eriksson et al., NeurIPS 2019). Needs only `numpy`, `scipy`, `torch` and `gpytorch`.

**How to use**
1. Set `FUNCTION_ID` (and optionally the other settings) in the next cell.
2. Run from the folder that contains `initial_data/`.
3. **Each week:** paste that week's new `weekN_X` / `weekN_Y` dictionaries into the data cell (the same lines you add to your pipeline) and re-run. The notebook detects the weeks automatically and rebuilds the TuRBO state from the full history, so nothing else carries over.

**Which functions is it worth running on?** Checked on the data through week 10 (refit before each weekly query, compare error with stated uncertainty):

| Function | Verdict |
|---|---|
| **5** (`USE_LOG = True`) | **Best candidate.** Well calibrated and noise-free; the global GP peak is an extrapolation far from your data, which is what a trust region guards against. |
| **8** | Works well (well calibrated, small upside left). |
| 1, 2, 3 | Not suitable: the GP attributes ~95% of the variation to noise. |
| 4, 6, 7 | Not yet: poorly calibrated (4, 7), noisy (6). |

The notebook re-checks this itself (sections 3 and 7) and prints a warning if the model for the chosen function is not trustworthy.

**What differs from the tutorial**

| Change | Reason (evidence from this project) |
|---|---|
| State is **replayed from the real query history** (exact repeats skipped) | The trust region only shrinks after `failure_tolerance` consecutive non-improving queries. Accidental repeats carry no information and must not count as failures. |
| Y is **standardised** inside the fit; optional **log-transform** | As in the scikit-learn pipeline. Log is needed for Function 5 (output spans ~48,000x). |
| Length-scales **bounded**, fit by **SciPy L-BFGS-B** (+3 restarts) | 200 Adam steps had not converged, and unbounded length-scales drift ever larger. SciPy is called directly because BoTorch's fit helper rejects plain GPyTorch models in recent versions. |
| Three **suggestion modes** | `centroid` = average of 200 Thompson picks (stable, but pure exploitation); `single_draw` = one random Thompson pick (the tutorial's behaviour, keeps exploration); `medoid` = the most typical of the 200 picks. |
| float64, seeded sampling, `randint(0, dim)` | Numerical stability, reproducibility, and a fix to a tutorial off-by-one. |

## Settings

In [1]:
FUNCTION_ID = 5          # 1..8
USE_LOG = None           # None = project default (log-transform for Function 5 only); True / False to override
MODE = "centroid"        # "centroid" (exploit) | "single_draw" (explore) | "medoid"
SEED = 0                 # used by MODE = "single_draw"; change it to get a different random draw
LS_BOUNDS = (0.05, 10.0) # GP length-scale bounds
NOISE_LB = 1e-6          # lower bound on the GP noise (in standardised units)
RUN_CALIBRATION_CHECK = True

In [2]:
import warnings
warnings.filterwarnings("ignore", module="gpytorch")
warnings.filterwarnings("ignore", module="linear_operator")   # benign "added jitter" notices when sampling 2000 candidates
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message=".*torch.jit.script.*deprecated.*")

import math
from dataclasses import dataclass

import numpy as np
import torch
import gpytorch
from gpytorch.constraints import Interval, GreaterThan
from torch.quasirandom import SobolEngine
from scipy.optimize import minimize

torch.set_default_dtype(torch.double)   # float64

## 1. Data
Paste new `weekN_X` / `weekN_Y` dictionaries at the bottom of this cell each week (same format as your pipeline: `{function_id: query}`).

In [ ]:
week1_X = {
    1: [0.5, 0.5],
    2: [0.816326, 0.959183],
    3: [0.406027, 0.410592, 0.49085],
    4: [0.394868, 0.425405, 0.352448, 0.464929],
    5: [0.323763, 0.780786, 0.952277, 0.952084],
    6: [0.345264, 0.328436, 0.412802, 0.727459, 0.096332],
    7: [0.057185, 0.424604, 0.394749, 0.207966, 0.379147, 0.653218],
    8: [0.158359, 0.089191, 0.006755, 0.06847, 0.779613, 0.339703, 0.109925, 0.073427],
}
week1_Y = {
    1: 2.6752879910742468e-09,
    2: -0.06913022970475102,
    3: -0.027253680430408482,
    4: -0.3932940148544044,
    5: 1703.1512205236875,
    6: -0.41036378008423247,
    7: 2.0679435112701356,
    8: 9.8674177220056,
}

week2_X = {
    1: [0.6, 0.3],
    2: [0.673469, 0.897959],
    3: [0.377732, 0.08922, 0.423517],
    4: [0.451289, 0.457435, 0.214355, 0.422482],
    5: [0.532623, 0.594318, 0.990109, 0.985016],
    6: [0.313541, 0.259869, 0.535241, 0.987761, 0.214623],
    7: [0.142611, 0.367475, 0.49149, 0.152728, 0.36191, 0.68565],
    8: [0.192332, 0.148385, 0.293081, 0.032111, 0.743423, 0.258716, 0.124238, 0.699129],
}
week2_Y = {
    1: 3.7759561298146587e-34,
    2: 0.4900359752987052,
    3: -0.0665391852481089,
    4: -3.2314120937555297,
    5: 1880.1864969021228,
    6: -0.6644176392227166,
    7: 2.3848190050793328,
    8: 9.8169820770504,
}

week3_X = {
    1: [0.7755, 0.7959],
    2: [0.6735, 0.9592],
    3: [0.6491, 0.9696, 0.0045],
    4: [0.3682, 0.4922, 0.4132, 0.4558],
    5: [0.5956, 0.9937, 0.9873, 0.9817],
    6: [0.2832, 0.5708, 0.0595, 0.96, 0.1099],
    7: [0.1018, 0.291, 0.5756, 0.016, 0.4824, 0.6257],
    8: [0.0955, 0.2689, 0.0107, 0.4009, 0.9253, 0.802, 0.0575, 0.7128],
}
week3_Y = {
    1: 5.706193265431438e-35,
    2: 0.5556031562372118,
    3: -0.12661734906119,
    4: -1.4732579244500381,
    5: 4397.233847397341,
    6: -1.141864063971705,
    7: 1.443148014381498,
    8: 9.739235081,
}

week4_X = {
    1: [0.0, 1.0],
    2: [0.6939, 0.1837],
    3: [0.3906, 0.6841, 0.0134],
    4: [0.4488, 0.3474, 0.4204, 0.4541],
    5: [0.803, 0.9455, 0.9975, 0.9763],
    6: [0.3663, 0.0835, 0.6415, 0.5525, 0.0403],
    7: [0.1377, 0.4569, 0.4845, 0.14, 0.3356, 0.8789],
    8: [0.0014, 0.2281, 0.078, 0.3066, 0.5973, 0.2664, 0.1899, 0.3868],
}
week4_Y = {
    1: 0.0,
    2: 0.4727543338128404,
    3: -0.09410459635472421,
    4: -0.27656674599950604,
    5: 4890.544342173683,
    6: -0.620316968433118,
    7: 1.4105773279212648,
    8: 9.861958861,
}

week5_X = {
    1: [0.84, 0.265],
    2: [0.7347, 0.102],
    3: [0.4159, 0.6786, 0.5489],
    4: [0.4974, 0.437, 0.3929, 0.4997],
    5: [0.9228, 0.9866, 0.9849, 0.8955],
    6: [0.4402, 0.3901, 0.6726, 0.6905, 0.0459],
    7: [0.1636, 0.5317, 0.5892, 0.1644, 0.2748, 0.6065],
    8: [0.1111, 0.1071, 0.1191, 0.0325, 0.533, 0.6719, 0.1267, 0.6119],
}
week5_Y = {
    1: 5.547267530712984e-124,
    2: 0.40787162317501413,
    3: -0.03123233828129471,
    4: -2.337100677643711,
    5: 5431.472348887776,
    6: -0.11438002459092875,
    7: 1.7026016586797958,
    8: 9.907796439,
}

week6_X = {
    1: [0.66, 0.69],
    2: [0.7143, 0.959],
    3: [0.5629, 0.5832, 0.5769],
    4: [0.3846, 0.429, 0.4098, 0.3929],
    5: [0.9228, 0.9866, 0.9849, 0.8955],
    6: [0.4469, 0.4699, 0.6443, 0.7291, 0.0035],
    7: [0.1682, 0.1899, 0.4607, 0.2863, 0.2475, 0.6452],
    8: [0.1873, 0.3589, 0.0359, 0.283, 0.6522, 0.6479, 0.0736, 0.2874],
}
week6_Y = {
    1: -0.00023213423561594152,
    2: 0.5925245614010894,
    3: -0.03300793803178473,
    4: 0.36745877084040446,
    5: 5431.472348887776,
    6: -0.2859671198094261,
    7: 3.0828540093223995,
    8: 9.822342154,
}

week7_X = {
    1: [0.6939, 0.7143],
    2: [0.6122, 0.1633],
    3: [0.0144, 0.0209, 0.612],
    4: [0.3846, 0.429, 0.4098, 0.3929],
    5: [0.95, 0.9838, 0.8664, 0.9096],
    6: [0.4402, 0.3901, 0.6726, 0.6905, 0.0459],
    7: [0.1392, 0.1763, 0.2522, 0.2448, 0.3108, 0.5756],
    8: [0.163, 0.0396, 0.0604, 0.2457, 0.7239, 0.2568, 0.2252, 0.6367],
}
week7_Y = {
    1: 1.3809715325920199e-08,
    2: 0.140680686233136,
    3: -0.14650531732960304,
    4: 0.36745877084040446,
    5: 4486.777815404592,
    6: -0.1607864395399717,
    7: 2.7391428761290046,
    8: 9.892736256,
}

week8_X = {
    1: [1.0, 1.0],
    2: [0.6735, 0.0816],
    3: [0.4098, 0.5249, 0.4771],
    4: [0.3771, 0.3295, 0.347, 0.4157],
    5: [0.9228, 0.9866, 0.9849, 0.8955],
    6: [0.458, 0.3427, 0.6298, 0.6189, 0.0906],
    7: [0.1778, 0.0215, 0.4483, 0.1805, 0.3655, 0.8406],
    8: [0.1214, 0.2627, 0.2572, 0.0453, 0.9581, 0.5578, 0.2206, 0.5654],
}
week8_Y = {
    1: 1.5141828121885905e-192,
    2: 0.5221448094518772,
    3: -0.0007266704057284045,
    4: 0.16685751397756876,
    5: 5431.472348887776,
    6: -0.23094225208206917,
    7: 2.1036531461866077,
    8: 9.910074099000001,
}

week9_X = {
    1: [0.6939, 0.7959],
    2: [0.9796, 0.7959],
    3: [0.2452, 0.4563, 0.4916],
    4: [0.3164, 0.4132, 0.3738, 0.3919],
    5: [0.9472, 0.9542, 0.9905, 0.8077],
    6: [0.3745, 0.2821, 0.6446, 0.7084, 0.017],
    7: [0.1242, 0.1453, 0.3665, 0.4159, 0.3319, 0.6466],
    8: [0.2737, 0.2604, 0.0739, 0.0786, 0.8623, 0.2307, 0.1087, 0.3526],
}
week9_Y = {
    1: -5.867451818923965e-23,
    2: -0.04650634345602017,
    3: -0.004879376674603271,
    4: -0.5864605987158771,
    5: 4563.687451278782,
    6: -0.26210707851655174,
    7: 2.6161329326338403,
    8: 9.815673679,
}

week10_X = {
    1: [0.3265, 0.1633],
    2: [0.7959, 0.8163],
    3: [0.3463, 0.5206, 0.4342],
    4: [0.4379, 0.3605, 0.43, 0.3237],
    5: [0.9873, 0.792, 0.9934, 0.9683],
    6: [0.3776, 0.4304, 0.838, 0.6628, 0.0234],
    7: [0.0414, 0.1739, 0.2592, 0.176, 0.2037, 0.769],
    8: [0.0503, 0.0629, 0.1873, 0.0325, 0.7434, 0.7233, 0.1361, 0.8361],
}
week10_Y = {
    1: 1.301679765856186e-53,
    2: 0.1930792757189555,
    3: -0.021101296700925382,
    4: -0.05043355027966312,
    5: 5197.049541931692,
    6: -0.5089339907588817,
    7: 1.8660100836123312,
    8: 9.898611879,
}

week11_X = {
    1: [0.3878, 0.0816],
    2: [0.7755, 0.2245],
    3: [0.0582, 0.4382, 0.5368],
    4: [0.4715, 0.3369, 0.3743, 0.3709],
    5: [0.9666, 0.9522, 0.957, 0.9547],
    6: [0.4343, 0.3412, 0.6075, 0.7019, 0.2012],
    7: [0.3592, 0.2205, 0.4918, 0.2916, 0.2329, 0.6164],
    8: [0.1041, 0.1625, 0.1219, 0.157, 0.791, 0.5024, 0.2202, 0.5199],
}
week11_Y = {
    1: 1.0281854721569829e-82,
    2: 0.2571610965236311,
    3: -0.022462140927196807,
    4: -0.870134160710037,
    5: 5856.10174723775,
    6: -0.2883720873482337,
    7: 2.682079613190043,
    8: 9.998060359,
}


In [4]:
DATA_DIR = "initial_data"
X0 = np.load(f"{DATA_DIR}/function_{FUNCTION_ID}/initial_inputs.npy")
Y0 = np.load(f"{DATA_DIR}/function_{FUNCTION_ID}/initial_outputs.npy")

week_ids = sorted(int(k[4:-2]) for k in list(globals())
                  if k.startswith("week") and k.endswith("_X") and k[4:-2].isdigit() and f"week{k[4:-2]}_Y" in globals())
if week_ids != list(range(1, len(week_ids) + 1)):
    print(f"WARNING: weeks found are {week_ids} (expected 1..{len(week_ids)} with no gaps)")
weekly_X = [list(globals()[f"week{w}_X"][FUNCTION_ID]) for w in week_ids]
weekly_Y = [float(globals()[f"week{w}_Y"][FUNCTION_ID]) for w in week_ids]

LOG_DEFAULTS = {5: True}
use_log = LOG_DEFAULTS.get(FUNCTION_ID, False) if USE_LOG is None else bool(USE_LOG)

X_np = np.vstack([X0, np.array(weekly_X)])
Y_np = np.concatenate([Y0, np.array(weekly_Y)])
if use_log and np.any(Y_np <= 0):
    raise ValueError(f"Function {FUNCTION_ID} has outputs <= 0, so a log-transform is not possible. Set USE_LOG = False.")

train_x = torch.tensor(X_np)
train_y = torch.tensor(Y_np)                                   # raw outputs (TuRBO's state uses these)
y_model = torch.log(train_y) if use_log else train_y           # what the GP is fitted to
DIM = train_x.shape[1]

print(f"Function {FUNCTION_ID}: {len(train_y)} observations ({len(Y0)} provided + {len(weekly_Y)} weekly), {DIM} dimensions")
print(f"weeks found: {week_ids[0]}..{week_ids[-1]} | log-transform: {use_log} | mode: {MODE}")
print(f"current best = {train_y.max().item():.6g} at {np.round(train_x[train_y.argmax()].numpy(), 3)}")

def describe(m, s):
    """Format a model-space prediction in the original output units."""
    return f"{math.exp(m):.4g} (1-sd range {math.exp(m - s):.4g} .. {math.exp(m + s):.4g})" if use_log else f"{m:.4f} +- {s:.4f}"

Function 5: 30 observations (20 provided + 10 weekly), 4 dimensions
weeks found: 1..10 | log-transform: True | mode: centroid
current best = 5431.47 at [0.923 0.987 0.985 0.896]


## 2. TuRBO state
Same dataclass and update rule as the tutorial.

In [5]:
@dataclass
class TurboState:
    dim: int
    batch_size: int = 1
    length: float = 0.8
    length_min: float = 0.5 ** 7
    length_max: float = 1.6
    failure_counter: int = 0
    failure_tolerance: int = float("nan")      # post-initialised
    success_counter: int = 0
    success_tolerance: int = 10                # tutorial value (paper uses 3)
    best_value: float = -float("inf")
    restart_triggered: bool = False

    def __post_init__(self):
        self.failure_tolerance = math.ceil(max([4.0 / self.batch_size, float(self.dim) / self.batch_size]))


def update_state(state, Y_next):
    Y_next = torch.as_tensor(Y_next).reshape(-1)
    if max(Y_next) > state.best_value + 1e-3 * math.fabs(state.best_value):   # needs a >0.1% improvement
        state.success_counter += 1
        state.failure_counter = 0
    else:
        state.success_counter = 0
        state.failure_counter += 1
    if state.success_counter == state.success_tolerance:       # expand
        state.length = min(2.0 * state.length, state.length_max)
        state.success_counter = 0
    elif state.failure_counter == state.failure_tolerance:     # shrink
        state.length /= 2.0
        state.failure_counter = 0
    state.best_value = max(state.best_value, max(Y_next).item())
    if state.length < state.length_min:
        state.restart_triggered = True
    return state

**Initialising the state.** The weekly queries were chosen by GP-UCB rather than TuRBO, but the success/failure signal still says how local progress is going, so the real history is replayed through `update_state`. A query is a *success* only if it beats the best by more than 0.1%. Exact repeats (same input **and** same output) are skipped: they carry no information. A repeat with a *different* output (a noisy function) is a real new measurement and does count.

In [6]:
def replay_state(Y_initial, weekly_X, weekly_Y, dim):
    state = TurboState(dim=dim, best_value=float(np.max(Y_initial)))
    seen = []
    for x, y in zip(weekly_X, weekly_Y):
        if any(np.allclose(x, sx, atol=1e-6) and abs(y - sy) < 1e-9 for sx, sy in seen):
            continue
        seen.append((x, y))
        state = update_state(state, torch.tensor([y]))
    return state

state = replay_state(Y0, weekly_X, weekly_Y, DIM)
print(state)
print(f"\n-> {state.failure_tolerance - state.failure_counter} more non-improving queries until the trust region halves "
      f"(length {state.length} -> {state.length / 2})")

TurboState(dim=4, batch_size=1, length=0.8, length_min=0.0078125, length_max=1.6, failure_counter=3, failure_tolerance=4, success_counter=0, success_tolerance=10, best_value=5431.472348887776, restart_triggered=False)

-> 1 more non-improving queries until the trust region halves (length 0.8 -> 0.4)


## 3. GP model
ARD kernel (one length-scale per input dimension): **a short length-scale means the output is sensitive to that input.** The fit also reports how much of the variation the model attributes to noise: if that is close to 1, the GP has found no usable structure and TuRBO has nothing reliable to work with.

In [7]:
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, ls_bounds=LS_BOUNDS):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()
        base = gpytorch.kernels.RBFKernel(ard_num_dims=train_x.shape[1],
                                          lengthscale_constraint=Interval(*ls_bounds))
        self.covar_module = gpytorch.kernels.ScaleKernel(base)

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(self.mean_module(x), self.covar_module(x))


def fit_gp(train_x, y, ls_bounds=LS_BOUNDS, noise_lb=NOISE_LB, n_restarts=3, seed=0, verbose=True):
    y = (y - y.mean()) / y.std()                               # standardise inside the fit
    lik = gpytorch.likelihoods.GaussianLikelihood(noise_constraint=GreaterThan(noise_lb))
    model = ExactGPModel(train_x, y, lik, ls_bounds)
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(lik, model)
    model.train(); lik.train()

    params = list(model.parameters())                          # includes the likelihood's noise parameter
    shapes, sizes = [p.shape for p in params], [p.numel() for p in params]

    def set_flat(v):
        i = 0
        for p, s, n in zip(params, shapes, sizes):
            p.data = torch.as_tensor(v[i:i + n], dtype=p.dtype).reshape(s)
            i += n

    def closure(v):                                            # loss and gradient, in the form SciPy wants
        set_flat(v); model.zero_grad()
        loss = -mll(model(train_x), y)
        loss.backward()
        return loss.item(), np.concatenate([p.grad.numpy().ravel() for p in params])

    x0 = np.concatenate([p.detach().numpy().ravel() for p in params])
    rng = np.random.default_rng(seed)
    starts = [x0] + [x0 + rng.normal(0, 1.0, x0.shape) for _ in range(n_restarts)]   # default start + random restarts
    best = None
    for v0 in starts:
        try:
            r = minimize(closure, v0, jac=True, method="L-BFGS-B", options=dict(maxiter=500))
        except Exception:
            continue
        if best is None or r.fun < best.fun:
            best = r
    set_flat(best.x)

    model.train(); lik.train()
    with torch.no_grad():
        lml = (mll(model(train_x), y) * len(y)).item()
    ls = model.covar_module.base_kernel.lengthscale.detach().squeeze().numpy().reshape(-1)
    if verbose:
        print(f"total log marginal likelihood = {lml:.2f}   noise = {lik.noise.item():.1e} (fraction of the variance; 1.0 = all noise)")
        print(f"ARD length-scales (dims 1..{len(ls)}):", np.round(ls, 2))
        print("most -> least influential dimensions:", [int(i) + 1 for i in np.argsort(ls)])
        if lik.noise.item() > 0.5:
            print("WARNING: the GP attributes most of the variation to noise, so it has found no usable structure for this function. "
                  "TuRBO's suggestion would not be reliable: prefer the main GP-UCB pipeline here.")
    return model, lik


def predict_model_space(model, x, y_train):
    # posterior mean and sd of the latent function at x, in the space the GP was fitted in (log space if use_log)
    mu, sd = y_train.mean().item(), y_train.std().item()
    model.eval()
    with torch.no_grad():
        p = model(torch.as_tensor(np.atleast_2d(x)))
    return p.mean.numpy() * sd + mu, p.variance.sqrt().numpy() * sd

model, lik = fit_gp(train_x, y_model)

total log marginal likelihood = -2.94   noise = 1.0e-06 (fraction of the variance; 1.0 = all noise)
ARD length-scales (dims 1..4): [1.66 0.36 0.85 0.58]
most -> least influential dimensions: [2, 4, 3, 1]


## 4. Candidate generation (Thompson sampling in the trust region)
Same logic as the tutorial's `generate_batch`, with seeding and an `n_draws` option (several posterior samples over one candidate set; `n_draws=1` is the tutorial). When `prob_perturb = min(20/dim, 1)` equals 1 (dim <= 20, i.e. every function here), every coordinate is perturbed: TuRBO is then Thompson sampling over Sobol points inside a box around the best point, with the box shaped by the length-scales.

In [8]:
def generate_batch(state, model, X, Y, n_draws=1, n_candidates=None, seed=None):
    assert X.min() >= 0.0 and X.max() <= 1.0 and torch.all(torch.isfinite(Y))
    dim = X.shape[-1]
    if n_candidates is None:
        n_candidates = min(5000, max(2000, 200 * dim))
    if seed is not None:
        torch.manual_seed(seed)

    x_center = X[Y.argmax(), :].clone()                         # trust region is centred on the best point
    weights = model.covar_module.base_kernel.lengthscale.squeeze().detach().reshape(-1)
    weights = weights / weights.mean()
    weights = weights / torch.prod(weights.pow(1.0 / len(weights)))
    tr_lb = torch.clamp(x_center - weights * state.length / 2.0, 0.0, 1.0)
    tr_ub = torch.clamp(x_center + weights * state.length / 2.0, 0.0, 1.0)

    sobol = SobolEngine(dim, scramble=True, seed=seed)
    pert = tr_lb + (tr_ub - tr_lb) * sobol.draw(n_candidates).to(X)
    prob_perturb = min(20.0 / dim, 1.0)
    mask = torch.rand(n_candidates, dim) <= prob_perturb
    ind = torch.where(mask.sum(dim=1) == 0)[0]
    mask[ind, torch.randint(0, dim, size=(len(ind),))] = 1
    X_cand = x_center.expand(n_candidates, dim).clone()
    X_cand[mask] = pert[mask]

    model.eval()
    with torch.no_grad():
        samples = model(X_cand).sample(torch.Size([n_draws]))   # n_draws samples from the GP posterior
        X_next = X_cand[samples.argmax(dim=1)]                  # each sample votes for its best candidate
    return X_next, (tr_lb, tr_ub)

## 5. The suggestion
- **`centroid`** (default): 200 Thompson picks (20 independent candidate sets x 10 posterior samples), averaged. Stable between runs, but averaging removes the randomness that gives Thompson sampling its exploration, so this is effectively *exploitation*. A guard falls back to the medoid if the centroid scores worse than the typical pick (the peak is on the edge of the trust region, or there are several peaks).
- **`single_draw`**: one random Thompson pick, the tutorial's behaviour. Keeps exploration; change `SEED` for a different draw.
- **`medoid`**: the most typical of the 200 picks.

In [9]:
def next_query_via_TuRBO(train_x, y_model, state, model, mode="centroid", seed=0, n_sets=20, n_draws=10):
    if mode == "single_draw":
        x, (lb, ub) = generate_batch(state, model, train_x, y_model, n_draws=1, seed=seed)
        return x[0].numpy(), x.numpy(), f"single Thompson draw (seed {seed})", (lb, ub)
    picks = []
    for k in range(n_sets):
        x, (lb, ub) = generate_batch(state, model, train_x, y_model, n_draws=n_draws, seed=k)
        picks.append(x.numpy())
    P = np.vstack(picks)
    d = np.linalg.norm(P[:, None, :] - P[None, :, :], axis=2)
    medoid = P[np.argmin(d.sum(axis=1))]
    if mode == "medoid":
        return medoid, P, "medoid of 200 Thompson picks", (lb, ub)
    centroid = P.mean(axis=0)
    pick_means, _ = predict_model_space(model, P, y_model)
    centroid_mean, _ = predict_model_space(model, centroid, y_model)
    if centroid_mean[0] >= np.median(pick_means):
        return centroid, P, "centroid of 200 Thompson picks", (lb, ub)
    return medoid, P, "MEDOID (the centroid scored below the typical pick: the peak may sit on the edge of the trust region, or the posterior may have several peaks)", (lb, ub)

x_next, picks, how, (tr_lb, tr_ub) = next_query_via_TuRBO(train_x, y_model, state, model, mode=MODE, seed=SEED)
Xn, Yn = train_x.numpy(), train_y.numpy()

print("trust region lower:", np.round(tr_lb.numpy(), 3))
print("trust region upper:", np.round(tr_ub.numpy(), 3))
if len(picks) > 1:
    print(f"average distance of the Thompson picks from their centroid: {np.linalg.norm(picks - picks.mean(0), axis=1).mean():.3f}  (a rough measure of how much exploration there is)")
print(f"\nSUGGESTED QUERY ({how}):")
print(np.round(x_next, 4).tolist())

d_prior = np.linalg.norm(Xn - x_next, axis=1).min()
print(f"\ndistance to nearest previous query: {d_prior:.3f}  ->  {'DUPLICATE!' if d_prior < 0.02 else 'ok, not a duplicate'}")
m, s = predict_model_space(model, x_next, y_model)
print(f"GP prediction at the suggestion: {describe(m[0], s[0])}   (current best {Yn.max():.6g})")

trust region lower: [0.021 0.792 0.522 0.58 ]
trust region upper: [1. 1. 1. 1.]
average distance of the Thompson picks from their centroid: 0.166  (a rough measure of how much exploration there is)

SUGGESTED QUERY (MEDOID (the centroid scored below the typical pick: the peak may sit on the edge of the trust region, or the posterior may have several peaks)):
[0.9666, 0.9522, 0.957, 0.9547]

distance to nearest previous query: 0.086  ->  ok, not a duplicate
GP prediction at the suggestion: 5401 (1-sd range 5061 .. 5763)   (current best 5431.47)


## 6. Cross-check: where does the GP itself say the peak is?
Independent of any sampling, maximise the GP posterior mean inside the trust region with a gradient optimiser (30 starts, including your best points). If the suggestion sits close to this point, the sampling noise has averaged out; if they are far apart, the answer is less clear-cut.

In [10]:
def maximise_posterior_mean(model, lb, ub, start_points, n_rand=25, seed=0):
    rng = np.random.default_rng(seed)
    model.eval()

    def loss_and_grad(v):
        x = torch.tensor(v, requires_grad=True)
        loss = -model(x.reshape(1, -1)).mean.sum()      # maximise the mean = minimise its negative
        loss.backward()
        return loss.item(), x.grad.numpy().copy()       # x.grad is already the gradient of the loss

    starts = [np.clip(p, lb, ub) for p in start_points] + [lb + (ub - lb) * rng.random(len(lb)) for _ in range(n_rand)]
    best = None
    for v0 in starts:
        r = minimize(loss_and_grad, v0, jac=True, method="L-BFGS-B", bounds=list(zip(lb, ub)))
        if best is None or r.fun < best.fun:
            best = r
    return best.x

best_points = [Xn[i] for i in np.argsort(-Yn)[:5]]
x_opt = maximise_posterior_mean(model, tr_lb.numpy(), tr_ub.numpy(), best_points)
m_opt, s_opt = predict_model_space(model, x_opt, y_model)
gap = np.linalg.norm(x_opt - x_next)
print("GP posterior-mean maximiser in the trust region:", np.round(x_opt, 4).tolist())
print(f"   GP value there = {describe(m_opt[0], s_opt[0])};  distance to the section-5 suggestion = {gap:.3f}")
print("   ->", "close: the suggestion is a stable summary of the peak" if gap < 0.15
      else "far apart: the posterior is spread out or has several peaks; with single_draw this is expected (that is its exploration)")

GP posterior-mean maximiser in the trust region: [1.0, 1.0, 1.0, 0.9055]
   GP value there = 6025 (1-sd range 5643 .. 6432);  distance to the section-5 suggestion = 0.088
   -> close: the suggestion is a stable summary of the peak


## 7. Sanity check: is this GP's uncertainty trustworthy? (forward-chaining)
The data are sequential, so the fair test is: refit on everything *before* each weekly query, predict it, and compare the error with the stated uncertainty (`z = error / sd`, where sd includes observation noise). If the model is honest, about 95% of `|z|` should be below 2. Computed in the space the GP is fitted in (log space for Function 5). Takes under a minute.

In [11]:
if RUN_CALIBRATION_CHECK:
    n0, n = len(Y0), len(train_y)
    zs, errs = [], []
    for k in range(n0, n):
        m_k, l_k = fit_gp(train_x[:k], y_model[:k], verbose=False)
        mu_k, sd_k = y_model[:k].mean().item(), y_model[:k].std().item()
        m_k.eval()
        with torch.no_grad():
            p = l_k(m_k(train_x[k].reshape(1, -1)))             # predictive distribution of a NEW observation (includes noise)
        pred, psd = p.mean.item() * sd_k + mu_k, p.variance.sqrt().item() * sd_k
        err = y_model[k].item() - pred
        zs.append(err / psd); errs.append(err)
        print(f"week {k - n0 + 1:2d}: actual {train_y[k].item():.6g}   predicted {describe(pred, psd)}   z = {err / psd:+.1f}")
    zs, errs = np.array(zs), np.array(errs)
    frac = np.mean(np.abs(zs) > 2)
    print(f"\nfraction with |z| > 2: {frac:.0%} (about 5% is well calibrated)   median |z|: {np.median(np.abs(zs)):.2f} (about 0.67 is ideal)   mean error: {errs.mean():+.4g}")
    if lik.noise.item() > 0.5:
        print("VERDICT: not usable. The model attributes most of the variation to noise, so it only looks 'calibrated' because its "
              "error bars are huge. It has not learned this function: prefer the main GP-UCB pipeline.")
    elif frac > 0.3:
        print("VERDICT: poorly calibrated. Treat this suggestion as a guess and prefer the main GP-UCB pipeline for this function.")
    else:
        print("VERDICT: calibration is acceptable.", end=" ")
        if errs.mean() < 0:
            print("Results came in slightly below predictions on average (the usual 'winner's curse' from always querying where the model is most optimistic).")
        else:
            print("Results came in above predictions on average: the model has been conservative, which is the safe direction for an optimiser.")

week  1: actual 1703.15   predicted 715.3 (1-sd range 162 .. 3158)   z = +0.6


week  2: actual 1880.19   predicted 684.5 (1-sd range 332.1 .. 1411)   z = +1.4


week  3: actual 4397.23   predicted 1086 (1-sd range 248.8 .. 4739)   z = +0.9


week  4: actual 4890.54   predicted 2669 (1-sd range 845.5 .. 8425)   z = +0.5


week  5: actual 5431.47   predicted 2547 (1-sd range 858.7 .. 7557)   z = +0.7


week  6: actual 5431.47   predicted 5431 (1-sd range 5412 .. 5451)   z = +0.0


week  7: actual 4486.78   predicted 4411 (1-sd range 3298 .. 5900)   z = +0.1


week  8: actual 5431.47   predicted 5431 (1-sd range 5414 .. 5449)   z = +0.0


week  9: actual 4563.69   predicted 2962 (1-sd range 2434 .. 3603)   z = +2.2


week 10: actual 5197.05   predicted 3492 (1-sd range 2323 .. 5251)   z = +1.0

fraction with |z| > 2: 10% (about 5% is well calibrated)   median |z|: 0.64 (about 0.67 is ideal)   mean error: +0.5486
VERDICT: calibration is acceptable. Results came in above predictions on average: the model has been conservative, which is the safe direction for an optimiser.


## 8. After you submit and get the result
1. Paste the new `weekN_X` / `weekN_Y` dictionaries into the data cell (section 1).
2. Re-run all cells. The state is replayed automatically: a >0.1% improvement counts as a success; otherwise the failure counter rises, and after `failure_tolerance` failures in a row the trust region halves.
3. If `state.restart_triggered` ever becomes `True`, the trust region has collapsed and TuRBO would restart with a fresh, wide region.
4. Record the **prediction** from section 5 next to the **actual** result. With one query per function per week you cannot tell whether TuRBO beats GP-UCB on a single function, but you *can* tell whether its predictions come true.